# Risk Scoring System — Phase 3: Scoring Engine Demonstration

The engine logic lives in three source files (keep them in your `src/` folder next to this notebook, or adjust the path in Cell 0):
- **config.py**  — tunable weights + thresholds, with reasoning for every weight
- **scorers.py** — one 0-100 scoring function per feature
- **engine.py**  — aggregates sub-scores into a final 0-100 anomaly score + breakdown

**What the score means (read this):** the output is an ANOMALY / DEVIATION score — how unusual a transaction is versus the account's own normal behaviour. It is **not** a fraud probability; this dataset has no fraud labels, so no probability claim is made.

This notebook:
1. loads the engine and profiles
2. scores some REAL transactions from the dataset (should mostly look normal)
3. scores deliberately constructed ANOMALOUS transactions (should score high)
4. shows the explainable breakdown for each

## 0. Setup — point Python at your src/ folder

In [ ]:
import sys, json
import pandas as pd

# Folder containing config.py, scorers.py, engine.py
SRC = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\src"
DATA = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\bank_transactions.csv"
PROFILES = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\account_profiles.json"

if SRC not in sys.path:
    sys.path.insert(0, SRC)

from engine import RiskEngine

engine = RiskEngine(PROFILES)
print(f"Engine loaded with {len(engine.profiles)} account profiles")

Engine loaded with 495 account profiles


## 1. Load data (to pull real transactions and compute txns_today)

In [ ]:
df = pd.read_csv(DATA)
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'], errors='coerce')
df['hour'] = df['TransactionDate'].dt.hour
df['date'] = df['TransactionDate'].dt.date

# txns per account per day, to supply 'txns_today' honestly
daily_counts = df.groupby(['AccountID','date']).size().rename('txns_today').reset_index()
df = df.merge(daily_counts, on=['AccountID','date'], how='left')
print(f"{len(df)} transactions ready")

## 2. Score REAL transactions from the data

These are genuine rows. Because the baseline was built from this same data, most should score LOW (they ARE the account's normal). That's the correct behaviour — a sanity check that the engine doesn't flag normal activity.

In [ ]:
def row_to_txn(r):
    return {
        "account_id":      r["AccountID"],
        "amount":          float(r["TransactionAmount"]),
        "location":        r["Location"],
        "merchant":        r["MerchantID"],
        "current_balance": float(r["AccountBalance"]),
        "txns_today":      int(r["txns_today"]),
        "login_attempts":  int(r["LoginAttempts"]),
        "hour":            int(r["hour"]),
    }

# pick a few real rows from accounts that have decent history
good_hist = df.groupby('AccountID').filter(lambda g: len(g) >= 6)
sample = good_hist.sample(5, random_state=42)

for _, r in sample.iterrows():
    res = engine.score_transaction(row_to_txn(r))
    print(f"Account {res['account_id']} | score {res['risk_score']:5.1f} | "
          f"{res['decision']:6} | conf={res['confidence']} | "
          f"baseline={res['baseline_txns']} txns")
print("\nReal transactions should mostly score low (they are the account's normal).")

## 3. Full breakdown for one real transaction\nShows the explainability — exactly which features contributed what.

In [ ]:
r = sample.iloc[0]
res = engine.score_transaction(row_to_txn(r))
print(f"Account     : {res['account_id']}")
print(f"Risk score  : {res['risk_score']}  (anomaly score, not fraud probability)")
print(f"Decision    : {res['decision']}")
print(f"Confidence  : {res['confidence']} — {res['confidence_note']}")
print("\nBreakdown (sorted by contribution):")
print(f"{'feature':10} {'sub':>6} {'weight':>7} {'contrib':>8}   reason")
for b in res['breakdown']:
    print(f"{b['feature']:10} {b['sub_score']:6.1f} {b['weight']:7.2f} "
          f"{b['contribution']:8.2f}   {b['reason']}")

## 4. Score DELIBERATELY ANOMALOUS transactions

Now we construct transactions that SHOULD score high, to prove the engine reacts correctly. We take a real account with good history and inject anomalies one at a time, so you can see each feature respond.

In [ ]:
# choose an account with the most history for a stable baseline
counts = df.groupby('AccountID').size().sort_values(ascending=False)
acct = counts.index[0]
profile = engine.profiles[acct]
print(f"Using account {acct} with {profile['txn_count']} txns")
print(f"  avg amount      : {profile['amt_mean']}")
print(f"  max amount      : {profile['amt_max']}")
print(f"  known locations : {profile['known_locations']}")
print(f"  known merchants : {len(profile['known_merchants'])} merchants")
print(f"  avg balance     : {profile['avg_balance']}")
print(f"  avg daily txns  : {profile['avg_daily_txns']}")

In [ ]:
def show(title, txn):
    res = engine.score_transaction(txn)
    print(f"\n=== {title} ===")
    print(f"Score {res['risk_score']} -> {res['decision']} (conf {res['confidence']})")
    for b in res['breakdown'][:4]:
        if b['contribution'] > 0:
            print(f"   {b['feature']:10} +{b['contribution']:.1f}  ({b['reason']})")

base = {
    "account_id": acct,
    "amount": profile['amt_mean'],
    "location": profile['known_locations'][0],
    "merchant": profile['known_merchants'][0],
    "current_balance": profile['avg_balance'],
    "txns_today": 1,
    "login_attempts": 1,
    "hour": 16,
}

# 4a: a normal-looking transaction (baseline control)
show("NORMAL transaction (control)", dict(base))

# 4b: huge amount
t = dict(base); t['amount'] = profile['amt_max'] * 3
show("ANOMALY: amount 3x the account's historical max", t)

# 4c: brand new location
t = dict(base); t['location'] = "Reykjavik"   # not a US city in the data
show("ANOMALY: never-seen location", t)

# 4d: new merchant
t = dict(base); t['merchant'] = "M999_unseen"
show("ANOMALY: never-seen merchant", t)

# 4e: balance drain
t = dict(base); t['amount'] = profile['avg_balance'] * 0.95; t['current_balance'] = profile['avg_balance']*0.05
show("ANOMALY: drains ~95% of available balance", t)

# 4f: velocity burst
t = dict(base); t['txns_today'] = int(profile['avg_daily_txns'] * 5) + 5
show("ANOMALY: 5x normal daily transaction volume", t)

# 4g: everything at once
t = {
    "account_id": acct,
    "amount": profile['amt_max'] * 3,
    "location": "Reykjavik",
    "merchant": "M999_unseen",
    "current_balance": profile['avg_balance']*0.05,
    "txns_today": int(profile['avg_daily_txns']*5)+5,
    "login_attempts": 4,
    "hour": 3,
}
show("ANOMALY: all signals at once (worst case)", t)

## 5. Honest reading of these results

Fill in after running:

- Did real transactions mostly score low? (they should)
- Did each injected anomaly raise the expected feature's contribution?
- Did "all signals at once" approach a high score / BLOCK?

**What this proves:** the engine correctly scores deviation from each account's
baseline, and every score is fully explainable via its breakdown.

**What this does NOT prove:** that high scores correspond to real fraud — we have
no fraud labels, so that claim is out of scope. The deliverable is a transparent
anomaly-scoring engine, which is exactly what was asked.

**Carried-forward limitations (state these honestly in the report):**
- 42% of accounts have <5 transactions -> thin baselines, flagged low-confidence.
- Time-of-day feature near-useless on this data (all txns 16-18h), weighted ~0.
- PreviousTransactionDate is an artifact; velocity derived from TransactionDate.

## 6. Next — Phase 4

Wrap this engine in a FastAPI service (`POST /score` -> risk score + breakdown),
mirroring the sentiment project's serving pattern, so a transaction can be scored
in real time over HTTP. Optionally a small dashboard to visualise score breakdowns.